<a href="https://colab.research.google.com/github/th100129/norajo/blob/main/spandata1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# pip install datasets transformers

import json
from typing import List, Dict, Tuple, Any

from datasets import load_dataset
from transformers import AutoTokenizer


# =========================================================
# 1) 설정
# =========================================================
DATASET_NAME = "klue"
DATASET_CONFIG = "ner"
TOKENIZER_NAME = "klue/bert-base"   # 예시. bert계열 아무거나 가능
MAX_LENGTH = 128
SAVE_TRAIN_PATH = "klue_ner_globalpointer_train.json"
SAVE_VALID_PATH = "klue_ner_globalpointer_valid.json"

LABEL_LIST = ["DT", "LC", "OG", "PS", "QT", "TI"]
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}


# =========================================================
# 2) BIO 태그 -> span 추출
#    입력:
#      tokens   : 문자 단위 리스트
#      ner_tags : 문자열 라벨 리스트 또는 정수 id 리스트
#    출력:
#      entities : [{"start": 시작문자인덱스, "end": 끝문자인덱스, "label": "PS", "text": "..."}]
# =========================================================
def bio_to_spans(tokens: List[str], ner_tags: List[Any], id2tag: Dict[int, str] = None) -> List[Dict[str, Any]]:
    """
    KLUE NER의 문자 단위 BIO 태그를 span 엔티티로 변환
    start/end는 '문자 인덱스' 기준, end는 inclusive
    """
    # ner_tags가 int면 문자열 태그로 변환
    if len(ner_tags) > 0 and isinstance(ner_tags[0], int):
        if id2tag is None:
            raise ValueError("ner_tags가 정수형이면 id2tag가 필요합니다.")
        tags = [id2tag[x] for x in ner_tags]
    else:
        tags = ner_tags

    entities = []
    start = None
    current_type = None

    for i, tag in enumerate(tags):
        if tag == "O":
            if current_type is not None:
                entities.append({
                    "start": start,
                    "end": i - 1,
                    "label": current_type,
                    "text": "".join(tokens[start:i])
                })
                start = None
                current_type = None
            continue

        prefix, ent_type = tag.split("-", 1)

        if prefix == "B":
            if current_type is not None:
                entities.append({
                    "start": start,
                    "end": i - 1,
                    "label": current_type,
                    "text": "".join(tokens[start:i])
                })
            start = i
            current_type = ent_type

        elif prefix == "I":
            # 비정상 BIO (예: I가 갑자기 시작) 보정
            if current_type is None:
                start = i
                current_type = ent_type
            elif current_type != ent_type:
                entities.append({
                    "start": start,
                    "end": i - 1,
                    "label": current_type,
                    "text": "".join(tokens[start:i])
                })
                start = i
                current_type = ent_type

    if current_type is not None:
        entities.append({
            "start": start,
            "end": len(tags) - 1,
            "label": current_type,
            "text": "".join(tokens[start:])
        })

    return entities


# =========================================================
# 3) 문자 인덱스 -> 토크나이저 토큰 인덱스 매핑
#    GlobalPointer는 보통 "토큰 시작/끝 인덱스"가 필요함
# =========================================================
def build_char_to_token_map(
    tokenizer,
    text: str,
    max_length: int
) -> Tuple[Dict[int, int], Dict[int, Tuple[int, int]], Dict[str, Any]]:
    """
    반환:
      char_to_token_start: 각 문자 인덱스 -> 해당 문자를 포함하는 첫 token index
      token_offset_map   : token index -> (char_start, char_end_exclusive)
      encoded            : tokenizer 결과
    """
    encoded = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_offsets_mapping=True
    )

    offsets = encoded["offset_mapping"]

    char_to_token = {}
    token_offset_map = {}

    for token_idx, (s, e) in enumerate(offsets):
        token_offset_map[token_idx] = (s, e)

        # special token은 보통 (0, 0)
        if s == e:
            continue

        for char_idx in range(s, e):
            # 같은 문자를 여러 서브토큰이 덮는 경우 거의 없지만, 첫 토큰 기준 유지
            if char_idx not in char_to_token:
                char_to_token[char_idx] = token_idx

    return char_to_token, token_offset_map, encoded


# =========================================================
# 4) 엔티티 span(문자 기준) -> 토큰 span으로 변환
# =========================================================
def char_span_to_token_span(
    char_start: int,
    char_end: int,   # inclusive
    char_to_token: Dict[int, int]
) -> Tuple[int, int]:
    """
    문자 span [char_start, char_end] -> 토큰 span [token_start, token_end]
    매핑 불가하면 (None, None)
    """
    if char_start not in char_to_token or char_end not in char_to_token:
        return None, None

    token_start = char_to_token[char_start]
    token_end = char_to_token[char_end]
    return token_start, token_end


# =========================================================
# 5) 한 샘플을 GlobalPointer 학습용 포맷으로 변환
# =========================================================
def convert_example(example, tokenizer, id2tag, max_length: int) -> Dict[str, Any]:
    """
    출력 예시:
    {
      "text": "...",
      "input_ids": [...],
      "attention_mask": [...],
      "token_type_ids": [...],  # 없을 수도 있음
      "entities": [
          {"start": 3, "end": 5, "label": "PS", "label_id": 3, "text": "홍길동"}
      ]
    }
    """
    text = "".join(example["tokens"])   # KLUE NER tokens는 문자 단위라 join 하면 원문 복원
    char_entities = bio_to_spans(example["tokens"], example["ner_tags"], id2tag=id2tag)

    char_to_token, token_offset_map, encoded = build_char_to_token_map(
        tokenizer=tokenizer,
        text=text,
        max_length=max_length
    )

    token_entities = []
    for ent in char_entities:
        if ent["label"] not in LABEL2ID:
            continue

        tok_start, tok_end = char_span_to_token_span(
            ent["start"], ent["end"], char_to_token
        )

        # truncation으로 잘린 경우 제외
        if tok_start is None or tok_end is None:
            continue

        token_entities.append({
            "start": tok_start,
            "end": tok_end,
            "label": ent["label"],
            "label_id": LABEL2ID[ent["label"]],
            "text": ent["text"],
            "char_start": ent["start"],
            "char_end": ent["end"]
        })

    result = {
        "text": text,
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "entities": token_entities
    }

    if "token_type_ids" in encoded:
        result["token_type_ids"] = encoded["token_type_ids"]

    return result


# =========================================================
# 6) 전체 split 변환
# =========================================================
def convert_split(dataset_split, tokenizer, id2tag, max_length: int) -> List[Dict[str, Any]]:
    converted = []
    for example in dataset_split:
        converted.append(
            convert_example(
                example=example,
                tokenizer=tokenizer,
                id2tag=id2tag,
                max_length=max_length
            )
        )
    return converted


# =========================================================
# 7) 저장
# =========================================================
def save_jsonl(data: List[Dict[str, Any]], path: str):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


# =========================================================
# 8) 실행
# =========================================================
def main():
    dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

    # HF datasets label names 가져오기
    features = dataset["train"].features
    ner_feature = features["ner_tags"].feature
    id2tag = {i: name for i, name in enumerate(ner_feature.names)}

    train_data = convert_split(dataset["train"], tokenizer, id2tag, MAX_LENGTH)
    valid_data = convert_split(dataset["validation"], tokenizer, id2tag, MAX_LENGTH)

    save_jsonl(train_data, SAVE_TRAIN_PATH)
    save_jsonl(valid_data, SAVE_VALID_PATH)

    print(f"train saved: {SAVE_TRAIN_PATH} ({len(train_data)}개)")
    print(f"valid saved: {SAVE_VALID_PATH} ({len(valid_data)}개)")

    # 샘플 확인
    print("\n=== sample ===")
    print(json.dumps(train_data[0], ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()
    import torch

def make_globalpointer_labels(entities, num_labels, seq_len):
    labels = torch.zeros((num_labels, seq_len, seq_len), dtype=torch.long)
    for ent in entities:
        start = ent["start"]
        end = ent["end"]
        label_id = ent["label_id"]

        if 0 <= start < seq_len and 0 <= end < seq_len and start <= end:
            labels[label_id, start, end] = 1
    return labels
    from torch.utils.data import Dataset
import torch
import json

class GlobalPointerNERDataset(Dataset):
    def __init__(self, path, num_labels, max_length):
        self.samples = []
        self.num_labels = num_labels
        self.max_length = max_length

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                self.samples.append(json.loads(line))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]

        input_ids = torch.tensor(item["input_ids"], dtype=torch.long)
        attention_mask = torch.tensor(item["attention_mask"], dtype=torch.long)

        if "token_type_ids" in item:
            token_type_ids = torch.tensor(item["token_type_ids"], dtype=torch.long)
        else:
            token_type_ids = torch.zeros_like(input_ids)

        labels = make_globalpointer_labels(
            entities=item["entities"],
            num_labels=self.num_labels,
            seq_len=len(item["input_ids"])
        )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "labels": labels,
            "text": item["text"],
            "entities": item["entities"]
        }

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

train saved: klue_ner_globalpointer_train.json (21008개)
valid saved: klue_ner_globalpointer_valid.json (5000개)

=== sample ===
{
  "text": "특히 영동고속도로 강릉 방향 문막휴게소에서 만종분기점까지 5㎞ 구간에는 승용차 전용 임시 갓길차로제를 운영하기로 했다.",
  "input_ids": [
    2,
    3727,
    30032,
    7825,
    4367,
    1091,
    2395,
    2198,
    2318,
    2024,
    27135,
    1038,
    2033,
    2377,
    2015,
    2532,
    2299,
    2118,
    25,
    3565,
    5757,
    2170,
    2259,
    8960,
    5119,
    5937,
    551,
    2454,
    2232,
    2200,
    2021,
    2138,
    3792,
    31302,
    2200,
    1902,
    2062,
    18,
    3,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
